# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the [Kilifi County Mental Health FAIR² dataset](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. The notebook follows a reproducible process powered by the Croissant schema and references entities (record sets, fields, columns) by their unique `@id` fields to ensure reliability and clarity.

### Dataset Source
The FAIR² dataset is defined via a Croissant schema and accessible at:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -U mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

We will examine the available record sets, obtain their `@id`s, and display the list of fields for each record set.

In [ ]:
# List all available record sets and their field @ids
record_sets = list(dataset.record_sets())

print("Available Record Sets and fields:")
record_set_ids = []
for rec_set in record_sets:
    print(f"- Record Set: name='{rec_set.name}', @id='{rec_set.id}'")
    record_set_ids.append(rec_set.id)
    print("  Fields (columns):")
    for field in rec_set.fields:
        print(f"    - name='{field.name}', @id='{field.id}', dataType='{field.data_type}'")
    print("")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis.

We build a dictionary of DataFrames keyed by their record set `@id`.

In [ ]:
# Extract data from each record set
# We use the record_set_ids obtained above so this code always works regardless of specific IDs
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records for record set @id='{recset_id}' ...")
    recs = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(recs)
    dataframes[recset_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

# As an example, choose the first record set for analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering, normalizing, and grouping.

You may adjust field choices depending on the record set.

In [ ]:
# Try to automatically pick a numeric field and a grouping field for demo. You may customize as needed.
# (For the Kilifi dataset, likely numeric fields include GAD-7, PHQ-9, PSQ; strings include age, gender, etc.)
import numpy as np

df = dataframes[main_record_set_id]

display_cols = df.columns.tolist()
print("Fields in this record set:", display_cols)

# Try several common names for a numeric field
possible_numeric_fields = [
    'GAD-7_total', 'PHQ-9_total', 'PSQ_total', 'gad7_score', 'phq9_score', 'psq_score', 'age', 'Age', 'psq', 'phq9', 'gad7', 'score', 'Score',
]
numeric_field = None
for c in display_cols:
    if c in possible_numeric_fields:
        if np.issubdtype(df[c].dtype, np.number):
            numeric_field = c
            break
    # Try to coerce to float
    try:
        df[c] = pd.to_numeric(df[c])
        if np.issubdtype(df[c].dtype, np.number):
            numeric_field = c
            break
    except Exception:
        continue

if numeric_field is None:
    # Just pick the first numeric field found
    for c in display_cols:
        try:
            df[c] = pd.to_numeric(df[c])
            if np.issubdtype(df[c].dtype, np.number):
                numeric_field = c
                break
        except Exception:
            continue

if numeric_field is None:
    raise ValueError("No numeric field found for EDA.")

print(f"Using numeric field: {numeric_field}")

# Filtering
threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalizing
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: try grouping by certain categorical field if present
possible_group_fields = ['gender', 'Gender', 'village', 'Village', 'education', 'Education', 'marital_status', 'Marital_Status']
group_field = None
for g in possible_group_fields:
    if g in df.columns:
        group_field = g
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions, such as the distribution of a key numeric score and (if available) how it varies by group.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Plot the distribution of the selected numeric field
plt.figure(figsize=(6,4))
df[numeric_field].hist(bins=20, color='skyblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# (If grouped data available) Bar plot of means by group
if group_field:
    grouped_df.plot.bar(x=group_field, y=numeric_field, legend=False, color='tomato')
    plt.title(f"Average {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the Kilifi County Mental Health FAIR² dataset using `mlcroissant`, explored its schema (record sets, fields), and performed basic exploratory data analysis (EDA) and visualization.

- **Schema-driven**: All entities and fields were referenced via `@id`, ensuring robustness for future reproducibility.
- **Data extraction and EDA**: We extracted record sets, selected relevant fields (like mental health assessment scores), and visualized distributions and group differences.
- **Potential for analysis**: This dataset supports deeper investigation into demographic and mental health patterns in Kilifi County, Kenya. Further analyses (missing data, psychometric properties, advanced models) can be implemented as needed.

For more on Croissant datasets and the `mlcroissant` library, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).
